<a href="https://colab.research.google.com/github/leonixxxxx/AI-Detect-Experiment/blob/main/02CLTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. INSTALL DEPENDENCIES & LOAD CLIP
# ==========================================
!pip install datasets tqdm -q
!pip install git+https://github.com/openai/CLIP.git -q

import torch
import torch.nn as nn
import torch.optim as optim
import clip
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
from PIL import Image
import io

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the CLIP ViT-L/14 model (This is what UniversalFakeDetect uses)
print("Loading CLIP ViT-L/14 Backbone...")
clip_model, clip_preprocess = clip.load("ViT-L/14", device=device)

# CRITICAL: Freeze the CLIP backbone so it doesn't lose its generalized knowledge
for param in clip_model.parameters():
    param.requires_grad = False

# ==========================================
# 2. BUILD THE CLASSIFIER HEAD
# ==========================================
# ViT-L/14 outputs a 768-dimensional feature vector.
# We just train a simple linear layer to map 768 -> 2 classes (Real vs AI)
embed_dim = clip_model.visual.output_dim
classifier = nn.Linear(embed_dim, 2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=0.001)

# ==========================================
# 3. PREPARE THE DATASET (Clean vs LINE)
# ==========================================
print("Loading Tiny-GenImage dataset...")
dataset = load_dataset("TheKernel01/Tiny-GenImage")

class SimulateLINECompression(object):
    def __init__(self, quality=70, max_size=1080):
        self.quality = quality
        self.max_size = max_size

    def __call__(self, img):
        img.thumbnail((self.max_size, self.max_size), Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=self.quality)
        buffer.seek(0)
        return Image.open(buffer).convert('RGB')

class GenImageDataset(Dataset):
    def __init__(self, hf_dataset, custom_transform, clip_preprocess):
        self.dataset = hf_dataset
        self.custom_transform = custom_transform
        self.clip_preprocess = clip_preprocess

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image = self.dataset[idx]['image'].convert('RGB')
        label = self.dataset[idx]['label']

        # Apply custom degradation (like LINE) if provided, then pass to CLIP's required preprocessor
        if self.custom_transform:
            image = self.custom_transform(image)
        image = self.clip_preprocess(image)
        return image, label

# Create DataLoaders
# We train ONLY on clean images, but evaluate on BOTH clean and LINE-degraded images
train_data = GenImageDataset(dataset['train'].select(range(2000)), None, clip_preprocess)
clean_val_data = GenImageDataset(dataset['validation'].select(range(500)), None, clip_preprocess)
line_val_data = GenImageDataset(dataset['validation'].select(range(500)), SimulateLINECompression(), clip_preprocess)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
clean_val_loader = DataLoader(clean_val_data, batch_size=32, shuffle=False)
line_val_loader = DataLoader(line_val_data, batch_size=32, shuffle=False)

# ==========================================
# 4. TRAIN THE LINEAR LAYER
# ==========================================
num_epochs = 3
print("\nStarting Training (Classifier Layer Only)...")
for epoch in range(num_epochs):
    classifier.train()
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)

        # Extract features using frozen CLIP
        with torch.no_grad():
            features = clip_model.encode_image(images).float()

        optimizer.zero_grad()
        outputs = classifier(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# ==========================================
# 5. EVALUATE ROBUSTNESS TO LINE COMPRESSION
# ==========================================
def evaluate(dataloader, desc):
    classifier.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            features = clip_model.encode_image(images).float()
            outputs = classifier(features)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100. * correct / total

print("\n--- Running UnivFD Robustness Test ---")
clean_acc = evaluate(clean_val_loader, "Testing Clean Images")
line_acc = evaluate(line_val_loader, "Testing LINE Images")

print("\n=== UNIV-FD RESULTS ===")
print(f"Accuracy on Clean Images:      {clean_acc:.2f}%")
print(f"Accuracy on LINE Images:       {line_acc:.2f}%")
print(f"Accuracy Drop (Degradation):   {(clean_acc - line_acc):.2f}%")

# Save your weights
torch.save(classifier.state_dict(), 'univfd_classifier.pth')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
Using device: cuda
Loading CLIP ViT-L/14 Backbone...


100%|████████████████████████████████████████| 890M/890M [00:09<00:00, 101MiB/s]


Loading Tiny-GenImage dataset...


README.md:   0%|          | 0.00/4.35k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  477MB            

...

Generating train split:   0%|          | 0/28000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7000 [00:00<?, ? examples/s]


Starting Training (Classifier Layer Only)...


Epoch 3/3: 100%|██████████| 63/63 [00:35<00:00,  1.80it/s]



--- Running UnivFD Robustness Test ---

=== UNIV-FD RESULTS ===
Accuracy on Clean Images:      81.40%
Accuracy on LINE Images:       84.20%
Accuracy Drop (Degradation):   -2.80%
